In [1]:
# Step 1: Install required packages
print("Installing packages (this may take 2-3 minutes)...")
print("   (Dependency warnings are normal in Colab and can be ignored)\n")
!pip install -q langchain langchain-community sentence-transformers chromadb transformers torch accelerate bitsandbytes 2>&1 | grep -v "dependency conflicts\|incompatible\|ERROR: pip's dependency" || true
print("\nInstallation complete!\n")

Installing packages (this may take 2-3 minutes)...
   (Dependency warnings are normal in Colab and can be ignored)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [2]:
# Step 2: Import libraries
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("Starting Complete RAG Pipeline Demo")
print("="*60)


Starting Complete RAG Pipeline Demo


In [3]:
# Step 3: Create sample documents (your knowledge base)
print("\nStep 1: Creating knowledge base...")
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, and chains.",
             metadata={"source": "doc1"}),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retrieved based on similarity, then an LLM generates an answer using those documents as context.",
             metadata={"source": "doc2"}),
    Document(page_content="Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning.",
             metadata={"source": "doc3"}),
    Document(page_content="ChromaDB is an open-source embedding database that works well with LangChain. It's lightweight and perfect for development.",
             metadata={"source": "doc4"}),
    Document(page_content="HuggingFace provides thousands of pre-trained models that can run locally or in the cloud. Popular models include FLAN-T5, GPT-2, and Llama.",
             metadata={"source": "doc5"}),
    Document(page_content="Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.",
             metadata={"source": "doc6"}),
]


Step 1: Creating knowledge base...


In [4]:
# Step 4: Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)
print(f"Created {len(splits)} document chunks")

Created 6 document chunks


In [5]:
# Step 5: Create embeddings
print("\nStep 2: Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")


Step 2: Loading embedding model...


/tmp/ipython-input-512036564.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


In [6]:
# Step 6: Create vector store
print("\n Step 3: Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="rag_demo"
)
print(" Vector database created")

# Step 7: Set up retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}  # Retrieve top 2 most relevant chunks
)


 Step 3: Creating vector database...
 Vector database created


In [7]:
# Step 8: Load LLM (using a small model that works in Colab)
print("\n Step 4: Loading Language Model...")
print("   (Using FLAN-T5 - large model)")

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)




 Step 4: Loading Language Model...
   (Using FLAN-T5 - large model)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [8]:
# Create pipeline directly (simpler and works better)
pipe = pipeline(
    "text2text-generation",
    model=model_name,
    tokenizer=tokenizer,
    max_new_tokens=256,  # Increased from max_length for better answers
    device=0 if torch.cuda.is_available() else -1,  # Use GPU if available
    model_kwargs={
        "temperature": 0.7,
        "do_sample": True,
        "top_p": 0.9,
    }
)

# Wrap in LangChain
llm = HuggingFacePipeline(pipeline=pipe)
print(" Language model loaded")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


 Language model loaded


/tmp/ipython-input-2766467586.py:16: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [9]:
# Step 9: Create custom prompt template
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context: {context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [10]:
# Step 10: Create RAG chain
print("\n Building RAG chain...")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)
print(" RAG chain ready!")


 Building RAG chain...
 RAG chain ready!


In [11]:
# Step 11: Test the complete RAG system
print("\n" + "="*60)
print(" COMPLETE RAG DEMONSTRATION")
print("="*60)

def ask_question(query):
    """
    Complete RAG pipeline: Retrieval + Augmented + Generation
    """
    print(f"\n Question: {query}")
    print("\n Step 1: Retrieving relevant documents...")

    result = qa_chain({"query": query})

    # Show retrieved documents
    print(f"Retrieved {len(result['source_documents'])} documents:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"\n   {i}. {doc.page_content[:150]}...")
        print(f"      Source: {doc.metadata['source']}")

    # Show generated answer
    print(f"\n Step 2: Generating answer using LLM...")
    print(f"\n Answer: {result['result']}")
    print("\n" + "-"*60)

# Run demo queries
print("\n Running Demo Queries...\n")

ask_question("What is RAG?")

ask_question("What are embeddings and why are they useful?")

ask_question("Which database should I use for vector storage?")



 COMPLETE RAG DEMONSTRATION

 Running Demo Queries...


 Question: What is RAG?

 Step 1: Retrieving relevant documents...


/tmp/ipython-input-3223637289.py:13: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": query})


Retrieved 2 documents:

   1. RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retriev...
      Source: doc2

   2. LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, and ch...
      Source: doc1

 Step 2: Generating answer using LLM...

 Answer: Retrieval-Augmented Generation.

------------------------------------------------------------

 Question: What are embeddings and why are they useful?

 Step 1: Retrieving relevant documents...
Retrieved 2 documents:

   1. Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings....
      Source: doc6

   2. Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning....
      Source: doc3

 Step 2: Generating answer using